## 4.4 可执行示例：FastGelu（regbase）

下面提供一个完整的可执行 FastGelu 示例，直接在 notebook 中编译运行。确保已执行环境准备（source `set_env.sh`）。

# 1.4 regbase 编程基础

regbase（Register-based，基于寄存器）是 Ascend C 为 Vector 计算单元提供的核心编程模型，也是生产级算子开发的标准方式。

---

## 学习目标

完成本节后，你将能够：
- 理解 regbase 编程模型的核心概念
- 了解 regbase 的流水线架构和关键元件
- 掌握 VF 寄存器的类型、操作流程和常用 API
- 掌握常用 VF 指令的使用方法

---

## 1. 什么是 regbase

### 1.1 概念

regbase 是一种**显式数据搬运 + 寄存器级 SIMD 计算**的编程模型。其核心思想是：

1. **数据搬运**：通过 DataCopy 将数据从 Global Memory 搬运到 AI Core 内部的 Unified Buffer（UB）
2. **寄存器计算**：在 UB 中分配 LocalTensor，通过 Vector Function（VF）指令进行 SIMD 并行计算
3. **结果写回**：计算完成后将结果从 UB 搬运回 Global Memory

### 1.2 dataflow

```
Global Memory (GM)               Unified Buffer (UB)            VF Reg（寄存器）
      │                                    │                         │
      │   DataCopy ───────── CopyIn ──────►│                         │
      │                                    │                         │
      │                                    ├── Load ────────────────►│
      │                                    │                         │
      │                                    │◄── Store ───────────────│
      │                                    │                         │
      │   DataCopy ◄───────── CopyOut ─────┤                         │
      │                                    │                         │
```

```mermaid
flowchart LR
    GM["Global Memory<br/>GM"] -->|"DataCopy<br/>CopyIn"| UB["Unified Buffer<br/>UB"]
    UB -->|"LoadAlign<br/>加载"| VF["VF Reg<br/>寄存器"]
    VF -->|"Neg/Exp/Add/Div<br/>SIMD 计算"| VF
    VF -->|"StoreAlign<br/>存储"| UB2["Unified Buffer<br/>UB"]
    UB2 -->|"DataCopy<br/>CopyOut"| GM2["Global Memory<br/>GM"]
```

### 1.3 regbase vs membase

| 特性 | membase | regbase |
|------|---------|----------|
| 数据访问 | `DataCopy` 搬运到 UB，`LocalTensor` 上 SIMD 计算 | `DataCopy` 搬运到 UB，`LoadAlign` 加载到 RegTensor 计算 |
| 编程复杂度 | 较简单（SIMD 在 UB 上操作，需管理 TPipe / TQue 流水） | 较复杂（需手动管理 TPipe / TQue 流水 + VF 寄存器） |
| 性能 | 中等（GM 延迟较高，硬加速掩盖部分开销） | 高（UB 低延迟 + 双缓冲隐藏搬运开销） |
| 适用场景 | 小数据量、快速原型验证 | 生产环境、大数据量、性能敏感场景 |

```mermaid
flowchart LR
    subgraph membase["membase 模式"]
        A1["DataCopy<br/>GM→UB"] --> A2["SIMD 指令<br/>UB上计算"]
        A2 --> A3["DataCopy<br/>UB→GM"]
    end
    subgraph regbase["regbase 模式"]
        B1["DataCopy<br/>GM→UB"] --> B2["LoadAlign<br/>UB→Reg"]
        B2 --> B3["VF 指令<br/>寄存器计算"]
        B3 --> B4["StoreAlign<br/>Reg→UB"]
        B4 --> B5["DataCopy<br/>UB→GM"]
    end
    membase --> regbase
```

---

## 2. regbase 架构

AI Core 内部采用流水线架构处理向量指令，由多个功能单元协作完成指令的取指、解码、分发和发射。

### 2.1 组成元件

| 元件 | 说明 |
|------|------|
| **Master Scalar** | 主标量核心，负责将指令发送到 MTE、VF 等各个流水线管道，同时控制程序执行流程（如循环和分支） |
| **VFQ**（Vector Function Queue） | 向量指令队列，包含 SFU（memory-to-memory 类指令，如 `MOV_UB_TO_UB`）、Vector Function 和向量预取指令 |
| **icache** | 指令缓存，Vector Thread 根据 VFQ 中指令提供的 PC 地址将指令取入 icache |
| **Instruction Buffer** | 指令缓冲区，icache 中的指令根据 HW Loop 控制逻辑进一步加载到此缓冲区 |
| **Decode** | 解码器，将指令解码为各自的控制信号并发送到下游，同时跟踪多个资源的可用性，每个周期最多向下游发送 6 条指令 |
| **Dispatch** | 分发器，将单条指令分发到对应流水线的指令队列中，核心功能之一是解析 load/store 指令的基地址（地址生成单元也在分发器中） |
| **Issue** | 发射器，根据各指令队列头部的就绪状态、处理元件的可用性、寄存器堆读写端口限制以及数据就绪性，将指令发射到对应的执行流水线 |

### 2.2 流水线行为

指令从取指到执行的完整流水线分为以下阶段：

```
Fetch → Dispatch1 → Decode → Rename → Dispatch2 → Issue
```

| 阶段 | 说明 |
|------|------|
| **Fetch** | Vector Thread 根据 VFQ 中指令提供的 PC 地址，将向量线程指令取到 icache 中。icache 中的指令根据 HW Loop Ctrl 逻辑进一步加载到 Instruction Buffer |
| **Dispatch1** | 将指令分解到辅助标量核心和实际矢量流水线中。所有分支/验证条件都在后续指令分发前被解决，保证被分发的指令一定会被执行 |
| **Decode** | 解码器向下游发送指令，将指令分解到 inner scalar 和 vector 流水线中。每周期最多发送 1 条标量指令；向量指令块发送到 OOO（乱序执行单元） |
| **Rename** | OOO 阶段先区分指令之间的寄存器依赖关系，在下一周期进行真正的寄存器重命名映射，并将指令块和寄存器 ID 发送给分发器 |
| **Dispatch2** | 将指令送入对应流水线的指令队列中，同时为装入（LSD）和存储（VST）指令计算基地址 |


```mermaid
flowchart LR
    FETCH["Fetch<br/>取指"] --> D1["Dispatch1<br/>指令分解"]
    D1 --> DEC["Decode<br/>解码"]
    DEC --> REN["Rename<br/>寄存器重命名"]
    REN --> D2["Dispatch2<br/>地址计算+分发"]
    D2 --> ISS["Issue<br/>发射"]
    ISS --> MTE["MTE 流水线<br/>数据搬运"]
    ISS --> VF["VF 流水线<br/>向量计算"]
    ISS --> CUBE["Cube 流水线<br/>矩阵计算"]
```
### 2.3 流水线示意图

<img src="./images/pipeline_overview.png" alt="pipeline overview" width="750px">

---

## 3. VF 寄存器

regbase 编程中直接操作的硬件寄存器包括 **RegTensor** 和 **MaskReg** 两类，由 `Reg::RegTrait` 控制寄存器数量。

### 3.1 寄存器总览

| 寄存器 | 类型定义 | 说明 |
|------|------|------|
| **RegTensor\<T\>** | `AscendC::Reg::RegTensor<T>` | 向量寄存器张量，容纳 `GetVecLen() / sizeof(T)` 个元素 |
| **MaskReg** | `AscendC::Reg::MaskReg` | 掩码寄存器，控制向量指令中哪些元素参与计算 |
| **RegTrait** | `Reg::RegTraitNumOne` / `Reg::RegTraitNumTwo` | 控制 RegTensor 使用的寄存器数量（1 或 2 个） |

### 3.2 RegTensor

RegTensor 是对一组向量寄存器的抽象，是 VF 计算的核心操作数：

```cpp
using namespace AscendC::Reg;

// 声明一个寄存器张量（默认使用 1 个寄存器）
RegTensor<float> srcReg;
RegTensor<half>  dstReg;

// 声明使用 2 个寄存器的张量
RegTensor<float, RegTraitNumTwo> largeReg;
```

**容量**：单个 RegTensor 能容纳的元素数由向量长度和数据类型决定，`GetVecLen()` 返回向量单元一次能处理的字节数。例如向量长度 256 字节时，`RegTensor<float>` 容纳 256 / 4 = 64 个 float。

支持的元素类型 `T`：

| 类型 | 位宽 | 说明 |
|------|------|------|
| `half` | 16-bit | FP16 |
| `bfloat16_t` | 16-bit | BF16 |
| `float` | 32-bit | FP32 |
| `int8_t` | 8-bit | INT8 |
| `int16_t` | 16-bit | INT16 |
| `int32_t` | 32-bit | INT32 |
| `int64_t` | 64-bit | INT64 |
| `uint8_t` | 8-bit | UINT8 |

### 3.3 MaskReg

MaskReg 是用于向量掩码操作的寄存器，控制向量指令中哪些元素生效：

```cpp
// 全开掩码：所有元素参与计算
MaskReg mask = CreateMask<float, MaskPattern::ALL>();

// 指定前 N 个元素参与
MaskReg partialMask = CreateMask<float, MaskPattern::SEQUENCE>();
```

### 3.4 寄存器操作流程

VF 计算的标准四步流程：

```
Load → Compute → Store → MemoryBarrier
```

```cpp
using namespace AscendC::Reg;

RegTensor<float> srcReg, dstReg;
MaskReg mask = CreateMask<float, MaskPattern::ALL>();

// Step 1: Load —— 从 UB 加载到寄存器
LoadAlign(srcReg, srcAddr);
LoadAlign(dstReg, dstAddr);

// Step 2: Compute —— 寄存器间计算
Add(dstReg, dstReg, srcReg, mask);

// Step 3: Store —— 从寄存器存回 UB
StoreAlign(dstAddr, dstReg, mask);

// Step 4: Memory Barrier —— 写入完成后再读取
LocalMemBar<MemType::VEC_STORE, MemType::VEC_LOAD>();
```

### 3.5 关键 API 速查

| API | 说明 |
|------|------|
| `LoadAlign(reg, addr)` | 对齐加载：UB → RegTensor |
| `StoreAlign(addr, reg, mask)` | 对齐存储：RegTensor → UB |
| `DataCopy(reg, addr)` | 通用搬入：UB → RegTensor（非对齐） |
| `DataCopy(addr, reg, mask)` | 通用搬出：RegTensor → UB（非对齐） |
| `CreateMask<T, Pattern>()` | 创建掩码寄存器 |
| `UpdateMask<T>(sreg)` | 根据标量值更新掩码 |
| `GetVecLen()` | 获取向量长度（字节） |
| `LocalMemBar<Before, After>()` | 内存屏障，保证 Before 类型指令完成后才执行 After 类型 |

---

## 4. VF 指令

### 4.1 算术运算

| 指令 | 原型 | 说明 |
|------|------|------|
| `Add` | `Add(dst, s1, s2, mask)` | 向量加法：dst = s1 + s2 |
| `Sub` | `Sub(dst, s1, s2, mask)` | 向量减法：dst = s1 - s2 |
| `Mul` | `Mul(dst, s1, s2, mask)` | 向量乘法：dst = s1 × s2 |
| `Div` | `Div(dst, s1, s2, mask)` | 向量除法：dst = s1 / s2 |
| `Mul` + `Add` | 组合使用 `Mul(dst, s1, s2, mask)` + `Add(dst, dst, s3, mask)` | 融合乘加：dst = s1 × s2 + s3 |
| `Adds` | `Adds(dst, src, scalar, mask)` | 标量-向量加法：dst = src + scalar |
| `Muls` | `Muls(dst, src, scalar, mask)` | 标量-向量乘法：dst = src × scalar |
| `Neg` | `Neg(dst, src, mask)` | 向量取反：dst = -src |
| `Abs` | `Abs(dst, src, mask)` | 向量绝对值：dst = \|src\| |
| `Reciprocal` | `Reciprocal(dst, src, mask)` | 向量倒数：dst = 1 / src |

### 4.2 数学函数

| 指令 | 原型 | 说明 |
|------|------|------|
| `Exp` | `Exp(dst, src, mask)` | e^x 指数 |
| `Log` | `Log(dst, src, mask)` | 自然对数 ln(x) |
| `Sqrt` | `Sqrt(dst, src, mask)` | 平方根 |
| `Rsqrt` | `Rsqrt(dst, src, mask)` | 倒数平方根（快速近似）|
| `Sin` | `Sin(dst, src, mask)` | 正弦 |
| `Cos` | `Cos(dst, src, mask)` | 余弦 |
| `Tanh` | `Tanh(dst, src, mask)` | 双曲正切 |

### 4.3 比较和选择

| 指令 | 原型 | 说明 |
|------|------|------|
| `Eq` | `Eq(mask, s1, s2, len)` | 逐元素比较相等 |
| `Gt` | `Gt(mask, s1, s2, len)` | 逐元素比较大于 |
| `Lt` | `Lt(mask, s1, s2, len)` | 逐元素比较小于 |
| `Select` | `Select(dst, mask, t, f, len)` | mask 为 true 选 t，否则选 f |

### 4.4 综合示例：FastGelu（regbase）

```cpp
// FastGelu (regbase): y = x / (1 + exp(-1.702 * x))
// 在 __simd_vf__ 函数中直接操作 RegTensor，遵循 Load → Compute → Store 流程
using namespace AscendC::Reg;

template <typename T>
__simd_vf__ inline void FastGeluVF(__ubuf__ T* xAddr, __ubuf__ T* yAddr,
                                     uint32_t vecLen, uint32_t repeat) {
    MaskReg mask = CreateMask<T, MaskPattern::ALL>();
    RegTensor<T> xReg, tReg;
    for (uint16_t i = 0; i < repeat; ++i) {
        LoadAlign(xReg, xAddr + i * vecLen);      // 1. Load: UB -> Reg
        Muls(tReg, xReg, 1.702f, mask);           // 2. t = 1.702 * x
        Neg(tReg, tReg, mask);                    // 3. t = -1.702 * x
        Exp(tReg, tReg, mask);                    // 4. t = exp(-1.702 * x)
        Adds(tReg, tReg, 1.0f, mask);             // 5. t = 1 + exp(-1.702 * x)
        Div(tReg, xReg, tReg, mask);              // 6. t = x / t
        StoreAlign(yAddr + i * vecLen, tReg, mask); // 7. Store: Reg -> UB
    }
    LocalMemBar<MemType::VEC_STORE, MemType::VEC_LOAD>();
}
```
## 小结

本节介绍了 regbase 编程基础：

1. **regbase 概念**：显式搬运 + 寄存器 SIMD 计算，GM → UB → VF Reg
2. **regbase 架构**：Master Scalar → VFQ → Fetch → Dispatch1 → Decode → Rename → Dispatch2 → Issue
3. **VF 寄存器**：RegTensor（向量寄存器张量）、MaskReg（掩码寄存器）、操作流程：Load → Compute → Store → MemoryBarrier
4. **VF 指令**：算术运算（Add/Mul/Div 等）、数学函数（Exp/Log/Sqrt）、比较选择（Eq/Gt/Select）

---

## 课后练习

1. 简述 regbase 模型的 GM → UB → VF Reg 三级数据流。
2. AI Core 指令流水线分为哪几个阶段？每个阶段的主要功能是什么？
3. RegTensor 和 MaskReg 的作用分别是什么？VF 计算的标准四步流程是什么？
4. 使用 VF 指令实现 GELU 激活函数：`y = 0.5 * x * (1 + tanh(x))`

---

上一节：[1.3 SIMD Membase 基础](./01.03_simd_membase_basics.ipynb)  
下一节：[1.5 SIMT 基础](./01.05_simt_basics.ipynb)

In [ ]:
!mkdir -p Sources/01.04

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("🎉 Environment initialization process completed successfully!")

In [ ]:
%%writefile Sources/01.04/fast_gelu_demo.asc

#include <cstdint>
#include <iostream>
#include <vector>
#include <cmath>
#include "acl/acl.h"
#include "kernel_operator.h"

using namespace AscendC;
using namespace AscendC::Reg;

constexpr uint32_t BUFFER_NUM = 2;
constexpr uint32_t TOTAL_LENGTH = 8 * 2048;
constexpr uint32_t TILE_NUM = 8;
constexpr uint32_t BLOCK_DIM = 8;

struct FastGeluArgs { uint32_t totalLength; uint32_t tileNum; };

template <typename T>
__simd_vf__ inline void FastGeluVF(__ubuf__ T* xAddr, __ubuf__ T* yAddr,
                                     uint32_t vecLen, uint32_t repeat) {
    MaskReg mask = CreateMask<T, MaskPattern::ALL>();
    RegTensor<T> xReg, tReg;
    for (uint16_t i = 0; i < repeat; ++i) {
        LoadAlign(xReg, xAddr + i * vecLen);
        Neg(tReg, xReg, mask);
        Exp(tReg, tReg, mask);
        Adds(tReg, tReg, 1.0f, mask);
        Div(tReg, xReg, tReg, mask);
        StoreAlign(yAddr + i * vecLen, tReg, mask);
    }
    LocalMemBar<MemType::VEC_STORE, MemType::VEC_LOAD>();
}

class KernelFastGelu {
public:
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y) {
        blockLength = TOTAL_LENGTH / BLOCK_DIM;
        tileLength = blockLength / TILE_NUM / BUFFER_NUM;
        vecLen = GetVecLen() / sizeof(float);
        uint32_t offset = blockLength * GetBlockIdx();
        xGm.SetGlobalBuffer((__gm__ float*)x + offset, blockLength);
        yGm.SetGlobalBuffer((__gm__ float*)y + offset, blockLength);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, tileLength * sizeof(float));
        pipe.InitBuffer(outQueueY, BUFFER_NUM, tileLength * sizeof(float));
    }
    __aicore__ inline void Process() {
        for (int32_t i = 0; i < TILE_NUM * BUFFER_NUM; i++)
            { CopyIn(i); Compute(i); CopyOut(i); }
    }
private:
    __aicore__ inline void CopyIn(int32_t p) {
        auto xL = inQueueX.AllocTensor<float>();
        DataCopy(xL, xGm[p * tileLength], tileLength);
        inQueueX.EnQue(xL);
    }
    __aicore__ inline void Compute(int32_t p) {
        auto xL = inQueueX.DeQue<float>();
        auto yL = outQueueY.AllocTensor<float>();
        FastGeluVF<float>((__ubuf__ float*)xL.GetPhyAddr(),
                          (__ubuf__ float*)yL.GetPhyAddr(),
                          vecLen, tileLength / vecLen);
        outQueueY.EnQue<float>(yL);
        inQueueX.FreeTensor(xL);
    }
    __aicore__ inline void CopyOut(int32_t p) {
        auto yL = outQueueY.DeQue<float>();
        DataCopy(yGm[p * tileLength], yL, tileLength);
        outQueueY.FreeTensor(yL);
    }
    TPipe pipe;
    TQue<TPosition::VECIN, BUFFER_NUM> inQueueX;
    TQue<TPosition::VECOUT, BUFFER_NUM> outQueueY;
    GlobalTensor<float> xGm, yGm;
    uint32_t blockLength, tileLength, vecLen;
};

__vector__ __global__ __aicore__ void fast_gelu(GM_ADDR x, GM_ADDR y, FastGeluArgs args) {
    KernelFastGelu op;
    op.Init(x, y);
    op.Process();
}

int32_t main() {
    aclInit(nullptr); aclrtSetDevice(0);
    aclrtStream stream; aclrtCreateStream(&stream);
    float *xHost, *yHost;
    aclrtMallocHost((void**)&xHost, TOTAL_LENGTH * sizeof(float));
    aclrtMallocHost((void**)&yHost, TOTAL_LENGTH * sizeof(float));
    for (uint32_t i = 0; i < TOTAL_LENGTH; i++)
        xHost[i] = -4.0f + i * 8.0f / TOTAL_LENGTH;
    void *xDev, *yDev;
    aclrtMalloc(&xDev, TOTAL_LENGTH * sizeof(float), ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc(&yDev, TOTAL_LENGTH * sizeof(float), ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMemcpy(xDev, TOTAL_LENGTH * sizeof(float), xHost,
                TOTAL_LENGTH * sizeof(float), ACL_MEMCPY_HOST_TO_DEVICE);
    FastGeluArgs args = {TOTAL_LENGTH, TILE_NUM};
    fast_gelu<<<BLOCK_DIM, nullptr, stream>>>((uint8_t*)xDev, (uint8_t*)yDev, args);
    aclrtSynchronizeStream(stream);
    aclrtMemcpy(yHost, TOTAL_LENGTH * sizeof(float), yDev,
                TOTAL_LENGTH * sizeof(float), ACL_MEMCPY_DEVICE_TO_HOST);
    bool pass = true;
    for (uint32_t i = 0; i < TOTAL_LENGTH; i++)
        if (fabsf(yHost[i] - xHost[i] / (1.0f + expf(-xHost[i]))) > 1e-4f)
            { pass = false; break; }
    printf("regbase FastGelu: %s\n", pass ? "PASSED" : "FAILED");
    aclrtFree(xDev); aclrtFree(yDev);
    aclrtFreeHost(xHost); aclrtFreeHost(yHost);
    aclrtDestroyStream(stream); aclrtResetDevice(0); aclFinalize();
    return 0;
}

以上代码包含完整的 VF 函数、Kernel 实现和 Host 调用，运行下面的编译命令即可验证。

In [ ]:
!bisheng Sources/01.04/fast_gelu_demo.asc --npu-arch=dav-3510 -o Sources/01.04/fast_gelu_demo -lm && echo "Build Successful"

In [ ]:
!./Sources/01.04/fast_gelu_demo

预期输出：`regbase FastGelu: PASSED`